# Runbook 4.4.a: Endpoint detection and response (EDR)  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 4.4.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** An EDR agent streams process-event telemetry from an endpoint. Each event records a process, its
parent process, whether it opened an external network connection, and how many bytes it
transferred. Build the analyzer that flags processes exhibiting either of the two classic
red flags an EDR platform's behavioral detection looks for: a "living off the land" parent-child
relationship, or unusual outbound data transfer.

**Lesson objective.** Describe how an EDR platform collects telemetry from a device, explain how it classifies process and file activity as suspicious or clean, and identify the core actions an EDR takes when it detects a threat.

### What comes in
A list of dicts, each with keys `pid` (int), `process` (str), `parent_process` (str),
`external_connection` (bool), `bytes_transferred` (int).

### What must come out
A sorted list of `pid` (int) for every event that is flagged.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
[{"pid": 501, "process": "powershell.exe", "parent_process": "winword.exe",
  "external_connection": True, "bytes_transferred": 500},
 {"pid": 502, "process": "chrome.exe", "parent_process": "explorer.exe",
  "external_connection": True, "bytes_transferred": 200000},
 {"pid": 503, "process": "backup.exe", "parent_process": "explorer.exe",
  "external_connection": True, "bytes_transferred": 5000000}]
    -> [501, 503]
```
*(pid 501 flagged for the living-off-the-land parent; pid 503 flagged for excessive outbound
transfer; pid 502 is a normal browser launch under the transfer threshold.)*

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Case-insensitive matching of process/parent names.
- [ ] Return a de-duplicated, ascending-sorted list of pids (each pid appears at most once even if
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(events: list) -> list
```

#### Input (`events`)
A list of dicts, each describing one EDR process event:

| key | type | meaning |
|---|---|---|
| `pid` | int | process id |
| `process` | str | the process image name, e.g. `"powershell.exe"` |
| `parent_process` | str | the parent process image name, e.g. `"winword.exe"` |
| `external_connection` | bool | whether the process opened a connection to an external host |
| `bytes_transferred` | int | total bytes transferred over that connection (0 if none) |

#### Output (`result`)
A de-duplicated, ascending-sorted list of `int` pids for every flagged event.

#### Detection rules (flag if EITHER fires)
1. `process.lower()` is in `{"powershell.exe", "cmd.exe", "wscript.exe", "cscript.exe",
   "mshta.exe"}` AND `parent_process.lower()` is not `"explorer.exe"`.
2. `external_connection` is `True` AND `bytes_transferred > 1000000`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- Process/parent name matching is case-insensitive.
- The output list has no duplicate pids and is sorted ascending.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
INTERPRETERS = {"powershell.exe", "cmd.exe", "wscript.exe", "cscript.exe", "mshta.exe"}
SAFE_PARENT = "explorer.exe"
TRANSFER_THRESHOLD = 1000000


def solve(events: list) -> list:
    flagged = set()
    for ev in events:
        pid = ev["pid"]
        process = ev.get("process", "").lower()
        parent = ev.get("parent_process", "").lower()
        external = ev.get("external_connection", False)
        transferred = ev.get("bytes_transferred", 0)

        if process in INTERPRETERS and parent != SAFE_PARENT:
            flagged.add(pid)
            continue
        if external and transferred > TRANSFER_THRESHOLD:
            flagged.add(pid)
            continue

    return sorted(flagged)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_1_prompt_scenario():
    events = [
        {"pid": 501, "process": "powershell.exe", "parent_process": "winword.exe",
         "external_connection": True, "bytes_transferred": 500},
        {"pid": 502, "process": "chrome.exe", "parent_process": "explorer.exe",
         "external_connection": True, "bytes_transferred": 200000},
        {"pid": 503, "process": "backup.exe", "parent_process": "explorer.exe",
         "external_connection": True, "bytes_transferred": 5000000},
    ]
    assert impl.solve(events) == [501, 503]


def test_example_2_no_flags():
    events = [
        {"pid": 1, "process": "notepad.exe", "parent_process": "explorer.exe",
         "external_connection": False, "bytes_transferred": 0},
        {"pid": 2, "process": "cmd.exe", "parent_process": "explorer.exe",
         "external_connection": False, "bytes_transferred": 0},
    ]
    assert impl.solve(events) == []


def test_edge_empty_events():
    assert impl.solve([]) == []


def test_case_insensitive_matching():
    events = [
        {"pid": 10, "process": "PowerShell.EXE", "parent_process": "WINWORD.EXE",
         "external_connection": False, "bytes_transferred": 0},
    ]
    assert impl.solve(events) == [10]


def test_interpreter_from_explorer_is_safe():
    events = [
        {"pid": 20, "process": "cmd.exe", "parent_process": "explorer.exe",
         "external_connection": False, "bytes_transferred": 0},
        {"pid": 21, "process": "CMD.EXE", "parent_process": "EXPLORER.EXE",
         "external_connection": False, "bytes_transferred": 0},
    ]
    assert impl.solve(events) == []


def test_transfer_threshold_boundary_not_exceeded():
    events = [
        {"pid": 30, "process": "sync.exe", "parent_process": "explorer.exe",
         "external_connection": True, "bytes_transferred": 1000000},
    ]
    assert impl.solve(events) == []


def test_transfer_threshold_boundary_exceeded():
    events = [
        {"pid": 31, "process": "sync.exe", "parent_process": "explorer.exe",
         "external_connection": True, "bytes_transferred": 1000001},
    ]
    assert impl.solve(events) == [31]


def test_no_external_connection_ignores_large_transfer():
    events = [
        {"pid": 40, "process": "backup.exe", "parent_process": "explorer.exe",
         "external_connection": False, "bytes_transferred": 9999999},
    ]
    assert impl.solve(events) == []


def test_result_deduplicated_and_sorted():
    events = [
        {"pid": 60, "process": "mshta.exe", "parent_process": "outlook.exe",
         "external_connection": True, "bytes_transferred": 5000000},
        {"pid": 50, "process": "wscript.exe", "parent_process": "outlook.exe",
         "external_connection": False, "bytes_transferred": 0},
    ]
    assert impl.solve(events) == [50, 60]


def test_living_off_the_land_variants():
    events = [
        {"pid": 71, "process": "cscript.exe", "parent_process": "svchost.exe",
         "external_connection": False, "bytes_transferred": 0},
        {"pid": 72, "process": "wscript.exe", "parent_process": "outlook.exe",
         "external_connection": False, "bytes_transferred": 0},
    ]
    assert impl.solve(events) == [71, 72]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([{"pid": 501, "process": "powershell.exe", "parent_process": "winword.exe",
  "external_connection": True, "bytes_transferred": 500},
 {"pid": 502, "process": "chrome.exe", "parent_process": "explorer.exe",
  "external_connection": True, "bytes_transferred": 200000},
 {"pid": 503, "process": "backup.exe", "parent_process": "explorer.exe",
  "external_connection": True, "bytes_transferred": 5000000}]),
     [501, 503]),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it